# EV Charging Station Planning & Demand Prediction System

# Notebook 03 : Data Cleaning & Preprocessing

## Business Objective

The objective of this notebook is to improve the quality of the collected datasets before performing exploratory data analysis and machine learning.

The cleaning process includes handling missing values, removing duplicate records, correcting data types, standardizing textual information, and preparing clean datasets for further analysis.

In [37]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

pd.set_option("display.max_columns",None)

In [38]:
stations_df = pd.read_csv("../data/raw/ev_charging_stations.csv")

usage_df = pd.read_csv("../data/raw/ev_charging_usage.csv")

registration_df = pd.read_csv("../data/raw/ev_registration_by_state.csv")

population_df = pd.read_csv("../data/raw/indian_cities_population.csv")

In [39]:
stations = stations_df.copy()

usage = usage_df.copy()

registration = registration_df.copy()

population = population_df.copy()

EV Charging Stations Cleaning

In [40]:
before = stations.shape[0]

stations = stations.drop_duplicates()

after = stations.shape[0]

print("Rows Before :", before)
print("Rows After  :", after)
print("Duplicates Removed :", before-after)

Rows Before : 855
Rows After  : 789
Duplicates Removed : 66


In [41]:
stations[stations["State"].isnull()]

,Station Name,City,State,Latitude,Longitude,Operator,Usage Type,Connector Type,Power (kW)
581,Muzhangodayil EVCS - ChargeMOD,Harippad,NaN,9.251674,76.487233,ChargeMod (IN),Public - Membership Required,CCS (Type 2),60.0


In [42]:
stations[stations["City"] == "Harippad"]

,Station Name,City,State,Latitude,Longitude,Operator,Usage Type,Connector Type,Power (kW)
581,Muzhangodayil EVCS - ChargeMOD,Harippad,NaN,9.251674,76.487233,ChargeMod (IN),Public - Membership Required,CCS (Type 2),60.0


In [43]:
stations.loc[
    stations["City"] == "Harippad",
    "State"
] = "Kerala"

In [44]:
stations[stations["Power (kW)"].isnull()]

,Station Name,City,State,Latitude,Longitude,Operator,Usage Type,Connector Type,Power (kW)
543,Gandhi Nagar KSEB EVCS - ChargeMOD,Arpookara,Kerala,9.632684,76.524402,ChargeMod (IN),Public - Membership Required,GB-T DC - GB/T 20234.3,NaN
805,Silavattam Village,"Madurathakam, Chengalpattu Dt",Tamil Nadu,12.481488,79.862522,(Unknown Operator),"Private - For Staff, Visitors or Customers",Unknown,NaN


In [45]:
stations[
    stations["Operator"] == "ChargeMod (IN)"
][["Station Name","Power (kW)"]]

,Station Name,Power (kW)
253,Roshi EVCS (EVOK) - ChargeMOD,30.0
254,Kanhangad KSEB EVCS - ChangeMOD,30.0
256,Kanhangad KSEB EVCS - ChangeMOD,30.0
257,Kanhangad KSEB EVCS - ChangeMOD,3.3
258,Ammas Fast - ChargeMOD,60.0
...,...,...
782,ChargeMOD EVCS Hub,30.0
783,Delta Power Kulathoor,60.0
785,Delta Power Kulathoor,30.0
786,KIMS Hospital Trivandrum (EVOK),30.0


In [46]:
stations.loc[
    stations["Station Name"]=="Gandhi Nagar KSEB EVCS - ChargeMOD",
    "Power (kW)"
]=30

In [47]:
stations["Power (kW)"].median()

np.float64(30.0)

In [48]:
stations["Power (kW)"] = stations["Power (kW)"].fillna(
    stations["Power (kW)"].median()
)

In [49]:
stations["City"] = stations["City"].str.strip().str.title()
stations["State"] = stations["State"].str.strip().str.title()
stations["Operator"] = stations["Operator"].str.strip()

In [50]:
stations.rename(columns={
    "Station Name": "station_name",
    "Power (kW)": "power_kw",
    "Connector Type": "connector_type",
    "Usage Type": "usage_type"
}, inplace=True)

In [51]:
print(stations.isnull().sum())
print(stations.duplicated().sum())
print(stations.shape)
print(stations.info())

station_name      0
City              0
State             0
Latitude          0
Longitude         0
Operator          0
usage_type        0
connector_type    0
power_kw          0
dtype: int64
0
(789, 9)
<class 'pandas.DataFrame'>
Index: 789 entries, 0 to 854
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   station_name    789 non-null    str    
 1   City            789 non-null    str    
 2   State           789 non-null    str    
 3   Latitude        789 non-null    float64
 4   Longitude       789 non-null    float64
 5   Operator        789 non-null    str    
 6   usage_type      789 non-null    str    
 7   connector_type  789 non-null    str    
 8   power_kw        789 non-null    float64
dtypes: float64(3), str(6)
memory usage: 136.6 KB
None


In [52]:
text_columns = [
    "Station_ID",
    "City",
    "Vehicle_Type",
    "Payment_Method"
]

for col in text_columns:
    usage[col] = usage[col].str.strip()

In [53]:
usage["Date"] = pd.to_datetime(usage["Date"])

In [54]:
usage["Charging_Start_Time"] = pd.to_datetime(
    usage["Charging_Start_Time"]
).dt.time

usage["Charging_End_Time"] = pd.to_datetime(
    usage["Charging_End_Time"]
).dt.time

In [55]:
usage.describe()

,Date,Energy_Consumed_kWh,Cost_INR
count,500,500.000000,500.00000
mean,2025-10-15 00:00:00,52.680400,526.80400
min,2025-10-15 00:00:00,3.200000,32.00000
25%,2025-10-15 00:00:00,28.650000,286.50000
50%,2025-10-15 00:00:00,51.600000,516.00000
75%,2025-10-15 00:00:00,76.225000,762.25000
max,2025-10-15 00:00:00,99.900000,999.00000
std,NaN,27.589008,275.89008


In [56]:
usage.rename(columns={
    "Station_ID": "station_id",
    "City": "city",
    "Date": "date",
    "Vehicle_Type": "vehicle_type",
    "Charging_Start_Time": "charging_start_time",
    "Charging_End_Time": "charging_end_time",
    "Energy_Consumed_kWh": "energy_consumed_kwh",
    "Cost_INR": "cost_inr",
    "Payment_Method": "payment_method"
}, inplace=True)

In [57]:
print(usage.isnull().sum())

print(usage.duplicated().sum())

print(usage.info())

station_id             0
city                   0
date                   0
vehicle_type           0
charging_start_time    0
charging_end_time      0
energy_consumed_kwh    0
cost_inr               0
payment_method         0
dtype: int64
0
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   station_id           500 non-null    str           
 1   city                 500 non-null    str           
 2   date                 500 non-null    datetime64[us]
 3   vehicle_type         500 non-null    str           
 4   charging_start_time  500 non-null    object        
 5   charging_end_time    500 non-null    object        
 6   energy_consumed_kwh  500 non-null    float64       
 7   cost_inr             500 non-null    float64       
 8   payment_method       500 non-null    str           
dtypes: datetime64[us](1), float64(2), o

In [58]:
registration.drop(columns=["Unnamed: 0"], inplace=True)

In [59]:
registration["State Name"] = (
    registration["State Name"]
    .str.strip()
    .str.title()
)

In [60]:
registration.rename(columns={
    "State Name": "state_name",
    "Two Wheeler": "two_wheeler",
    "Three Wheeler": "three_wheeler",
    "Four Wheeler": "four_wheeler",
    "Goods Vehicles": "goods_vehicles",
    "Public Service Vehicle": "public_service_vehicle",
    "Special Category Vehicles": "special_category_vehicles",
    "Ambulance/Hearses": "ambulance_hearses",
    "Construction Equipment Vehicle": "construction_equipment_vehicle",
    "Other": "other",
    "Grand Total": "grand_total",
    "total-charging-stations": "total_charging_stations"
}, inplace=True)

In [61]:
print(registration.isnull().sum())

print(registration.duplicated().sum())

registration.info()

state_name                        0
two_wheeler                       0
three_wheeler                     0
four_wheeler                      0
goods_vehicles                    0
public_service_vehicle            0
special_category_vehicles         0
ambulance_hearses                 0
construction_equipment_vehicle    0
other                             0
grand_total                       0
total_charging_stations           8
dtype: int64
0
<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   state_name                      32 non-null     str    
 1   two_wheeler                     32 non-null     int64  
 2   three_wheeler                   32 non-null     float64
 3   four_wheeler                    32 non-null     int64  
 4   goods_vehicles                  32 non-null     float64
 5   public_service_vehicle       

In [62]:
text_columns = [
    "name_of_city",
    "state_name",
    "location"
]

for col in text_columns:
    population[col] = population[col].str.strip().str.title()

In [63]:
population.rename(columns={
    "0-6_population_total": "population_0_6_total",
    "0-6_population_male": "population_0_6_male",
    "0-6_population_female": "population_0_6_female"
}, inplace=True)

In [64]:
print(population.isnull().sum())

print(population.duplicated().sum())

population.info()

name_of_city                      0
state_code                        0
state_name                        0
dist_code                         0
population_total                  0
population_male                   0
population_female                 0
population_0_6_total              0
population_0_6_male               0
population_0_6_female             0
literates_total                   0
literates_male                    0
literates_female                  0
sex_ratio                         0
child_sex_ratio                   0
effective_literacy_rate_total     0
effective_literacy_rate_male      0
effective_literacy_rate_female    0
location                          0
total_graduates                   0
male_graduates                    0
female_graduates                  0
dtype: int64
0
<class 'pandas.DataFrame'>
RangeIndex: 493 entries, 0 to 492
Data columns (total 22 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          -

In [65]:
stations.to_csv("../data/processed/clean_ev_charging_stations.csv", index=False)

usage.to_csv("../data/processed/clean_ev_charging_usage.csv", index=False)

registration.to_csv("../data/processed/clean_ev_registration.csv", index=False)

population.to_csv("../data/processed/clean_population.csv", index=False)